In [4]:
import math
def entropy(b, size):

    d = {}
    for i in range(0, len(b), size):
        block = b[i:i+size]
        d[block] = d.get(block, 0) + 1
    
    H = 0
    lenght = len(b) // size

    for i in d.values():
        if i > 0:
            p_i = i / lenght
            H -= p_i * math.log2(p_i)

    return H

print(entropy(b'abc', 1))
print(entropy(b'aaaa', 1))
print(entropy(b'aabb', 1))


1.584962500721156
0.0
1.0


In [6]:
def mtf_encode(b):
    alphabet = list(range(256))
    result = []

    for byte in b:
        index = alphabet.index(byte)
        result.append(index)
        alphabet.pop(index)
        alphabet.insert(0, byte)
    
    return bytes(result)

def mtf_decode(b):
    alphabet = list(range(256))
    result = []
    
    for index in b:
        byte = alphabet[index]
        result.append(byte)
        alphabet.pop(index)
        alphabet.insert(0, byte)
    
    return bytes(result)

original = b"aaaaaaaaaaaabbbbbbbcccccc"
encoded = mtf_encode(original)
decoded = mtf_decode(encoded)

print(f"Оригинал:  {original}")
print(f"MTF-код:   {encoded}")
print(f"Восстанов: {decoded}")

Оригинал:  b'aaaaaaaaaaaabbbbbbbcccccc'
MTF-код:   b'a\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00b\x00\x00\x00\x00\x00\x00c\x00\x00\x00\x00\x00'
Восстанов: b'aaaaaaaaaaaabbbbbbbcccccc'


In [ ]:
import heapq
def frequency_model(b):
    d = {}
    for i in b:
        d[i] = d.get(i, 0) + 1
    return d

def huffman_encode(data, model):
    # Сортируем
    queue = sorted([[char, freq] for char, freq in model.items()], key=lambda x: x[1])
    
    # Строим дерево
    while len(queue) > 1:
        one = queue.pop(0)
        two = queue.pop(0)
        parent = [None, one[1] + two[1], one, two]
        
        insert_position = 0
        for i, item in enumerate(queue):
            if parent[1] > item[1]:
                insert_position = i + 1
            else:
                break
        queue.insert(insert_position, parent)
    
    tree = queue[0] if queue else None
    
    # Кодируем
    codes = {}
    if tree:
        build_codes(tree, "", codes)
    
    # Кодируем данные
    encoded = ''.join(codes[char] for char in data)
    
    return codes, encoded, tree


def build_codes(node, prefix, codes):
    if len(node) == 2: # лист
        codes[node[0]] = prefix if prefix else "0"
        return

    if len(node) == 4: # узел
        build_codes(node[2], prefix + "0", codes)
        build_codes(node[3], prefix + "1", codes)


def huffman_decode(encoded, tree):
    if not tree or not encoded:
        return b""
    
    result = []
    node = tree
    for bit in encoded:
        if bit == '0':
            node = node[2]
        else:
            node = node[3]
        
        if len(node) == 2:
            result.append(node[0])
            node = tree
    
    return bytes(result)

def bits_to_bytes(bit_string):
    # Добавляем нули до кратности 8
    padding = (8 - len(bit_string) % 8) % 8
    bit_string += '0' * padding
    
    result = []
    for i in range(0, len(bit_string), 8):
        byte_chunk = bit_string[i:i+8]
        result.append(int(byte_chunk, 2))
    
    return bytes(result), padding


def bytes_to_bits(data, total_bits):
    bit_string = ''.join(f'{byte:08b}' for byte in data)
    return bit_string[:total_bits]

def read_huffman_file(filename):
    with open(filename, 'rb') as f:
        # Количество уникальных символов
        num_symbols = f.read(1)[0]
        
        # Таблица кодов
        codes = {}
        for _ in range(num_symbols):
            byte = f.read(1)[0]
            code_len = f.read(1)[0]
            code_byte = f.read(1)[0]
            code = f'{code_byte:08b}'[:code_len]
            codes[byte] = code
        
        # Служебная информация
        padding = f.read(1)[0]
        total_bits = int.from_bytes(f.read(4), 'big')
        
        # Сжатые данные
        compressed_data = f.read()
        encoded_bits = bytes_to_bits(compressed_data, total_bits)
    
    return codes, encoded_bits

def write_huffman_file(filename, codes, encoded_bits):
    compressed_bytes, padding = bits_to_bytes(encoded_bits)
    
    with open(filename, 'wb') as f:
        # Количество уникальных символов
        f.write(bytes([len(codes)]))

        # Таблица кодов
        for byte, code in codes.items():
            f.write(bytes([byte]))          # символ
            f.write(bytes([len(code)]))     # длина кода
            code_padded = code.ljust(8, '0')
            f.write(bytes([int(code_padded, 2)]))  # код
        
        # Служебная информация
        f.write(bytes([padding]))
        f.write(len(encoded_bits).to_bytes(4, 'big'))
        
        # Сжатые данные
        f.write(compressed_bytes)
    
    print(f"Записано в {filename}: {len(compressed_bytes) + 7 + len(codes)*3} байт")

def compress_file(input_path, output_path):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    model = frequency_model(data)
    codes, encoded_bits, tree = huffman_encode(data, model)
    write_huffman_file(output_path, codes, encoded_bits)
    
    print(f"Оригинал: {len(data)} байт")


def decompress_file(input_path, output_path):
    codes, encoded_bits = read_huffman_file(input_path)
    
    # Восстанавливаем дерево из кодов
    reverse_codes = {v: k for k, v in codes.items()}
    
    result = []
    buffer = ""
    for bit in encoded_bits:
        buffer += bit
        if buffer in reverse_codes:
            result.append(reverse_codes[buffer])
            buffer = ""
    
    with open(output_path, 'wb') as f:
        f.write(bytes(result))
    
    print(f"Распаковано: {len(result)} байт")


Оригинал: b'abracadabra'
Коды: {97: '0', 114: '10', 99: '1100', 100: '1101', 98: '111'}
Биты: 01111001100011010111100
Восстановлено: b'abracadabra'
Совпадает: True

Записано в test_compressed.bin: 18 байт
Оригинал: 9 байт
Распаковано: 9 байт
Файлы совпадают: True


In [27]:
def bwt_encode(b):
    n = len(b)
    matrix = [b[i:] + b[:i] for i in range(n)]
    matrix.sort()
    orig_index = matrix.index(b)
    last_column = bytes([i[-1] for i in matrix])

    return last_column, orig_index

def bwt_decode(b, index):
    n = len(b)
    matrix = [b""]*n

    for i in range(n):
        matrix = sorted([bytes([b[i]]) + matrix[i] for i in range(n)])

    return matrix[index]

original = b"0x62/0x61/0x6e/0x61/0x6e/0x61"
print(original)

encoded, idx = bwt_encode(original)
print(f"bwt: {encoded}, индекс: {idx}")

decoded = bwt_decode(encoded, idx)
print(decoded)

b'0x62/0x61/0x6e/0x61/0x6e/0x61'
bwt: b'2ee11///1//6666xxxxxx66000000', индекс: 8
b'0x62/0x61/0x6e/0x61/0x6e/0x61'


In [28]:
def bwt_decode_optimized(last_column, index):
    n = len(last_column)

    count = [0]*256
    for byte in last_column:
        count[byte] += 1
    
    total = 0
    for i in range(256):
        c = count[i]
        count[i] = total
        total += c
    
    next_pos = [0]*n
    current_count = [0]*256
    for i in range(n):
        byte = last_column[i]
        pos = count[byte] + current_count[byte]
        next_pos[i] = pos
        current_count[byte] += 1
    
    result = bytearray()
    row = index
    for i in range(n):
        byte = last_column[row]
        result.append(byte)
        row = next_pos[row]
    
    return bytes(result[::-1])

original = bytes([0x62, 0x61, 0x6E, 0x61, 0x6E, 0x61])
print(original)

encoded, idx = bwt_encode(original)
print(f"bwt: {encoded}, индекс: {idx}")

decoded = bwt_decode_optimized(encoded, idx)
print(decoded)

b'banana'
bwt: b'nnbaaa', индекс: 3
b'banana'


In [29]:
import struct
def bwt_encode_block(b):
    n = len(b)
    matrix = [b[i:] + b[:i] for i in range(n)]
    matrix.sort()
    orig_index = matrix.index(b)
    last_column = bytes([i[-1] for i in matrix])

    return last_column, orig_index

def bwt_decode_block(last_column, index):
    n = len(last_column)

    count = [0]*256
    for byte in last_column:
        count[byte] += 1
    
    total = 0
    for i in range(256):
        c = count[i]
        count[i] = total
        total += c
    
    next_pos = [0]*n
    current_count = [0]*256
    for i in range(n):
        byte = last_column[i]
        pos = count[byte] + current_count[byte]
        next_pos[i] = pos
        current_count[byte] += 1
    
    result = bytearray()
    row = index
    for i in range(n):
        byte = last_column[row]
        result.append(byte)
        row = next_pos[row]
    
    return bytes(result[::-1])

def bwt_encode(data, block_size=None):
    if block_size is None:
        return bwt_encode_block(data)
    
    num_blocks = (len(data) + block_size - 1) // block_size
    output = struct.pack('<I', num_blocks)
    
    for i in range(0, len(data), block_size):
        chunk = data[i:i+block_size]
        col, idx = bwt_encode_block(chunk)
        output += struct.pack('<I', idx)
        output += struct.pack('<I', len(col))
        output += col
        
    return output

def bwt_decode(data, index=None, blocked=False):
    if not blocked:
        return bwt_decode_block(data, index)
    
    offset = 0
    num_blocks = struct.unpack('<I', data[offset:offset+4])[0]
    offset += 4
    
    result = bytearray()
    for _ in range(num_blocks):
        idx = struct.unpack('<I', data[offset:offset+4])[0]
        length = struct.unpack('<I', data[offset+4:offset+8])[0]
        offset += 8
        
        block_data = data[offset:offset+length]
        offset += length
        
        result.extend(bwt_decode_block(block_data, idx))
        
    return bytes(result)